# Lab 8: Pick and Place with Vision and a Suction Gripper (worked solutions)

Companion to `PickAndPlaceTemplate.ipynb` and `pdfs/Pick_And_Place.pdf` (April version; the September
`Pick_And_Place-2.pdf` is Lab 9, see `Lab9_PickAndPlace_MoveIt_solution.ipynb`). Whether Lab 8 is still required is an open question
for the TAs (`gameplan.md` §11).

**How to read this notebook.** The code cells are the template's own cells with the blanks filled in: same order, same names,
nothing extra. Around them: the **Task**, **How it works** (each piece of syntax, each value and why), and short **Example**
blocks (code with its output, written as text). **How to run the whole system on the Pi and the laptop, which files go where,
and which values you can change:** `Lab8_Lab9_run_guide.md` in this folder.

**What the system does.** A *vision node* finds coloured cubes with a camera (YOLO), computes their 3D position in the
robot base frame and publishes it on `/detected_objects`. A *motion node* subscribes, filters, buffers, selects a target
(highest stack, then colour priority), and runs an eight-step pick and place with a vacuum gripper, sorting the cube into a
bin by colour. This notebook rebuilds the pieces of the motion node and the geometry of the vision node.

## Safe to run anywhere
| switch | default | meaning |
|---|---|---|
| `USE_REAL_HARDWARE` | `False` | `False`: `sim_hardware.install()` puts stand-ins in place of `pymycobot`, `RPi.GPIO` and `time.sleep`: every robot and pin action is printed with a simulated time and nothing moves. `True` (only on the Pi): the template's own `from pymycobot.mycobot import MyCobot` and `import RPi.GPIO as GPIO` load the real libraries. The serial notebook and `mycobot_control` must **not** use the port. |
| `enable_motion`, `enable_pump_test` | `= SIMULATED` | In simulation they are `True`, so the logic runs. On the real robot they are `False` until **a person** sets them to `True` after clearing the workspace and checking the printed targets. |


## Setup

In [1]:
%matplotlib widget
import os
import time
import numpy as np
import matplotlib.pyplot as plt

USE_REAL_HARDWARE = False   # True only on the Pi: real pymycobot + RPi.GPIO
SIMULATED = not USE_REAL_HARDWARE
if SIMULATED:
    import sim_hardware     # solutions/sim_hardware.py: stand-ins for pymycobot and RPi.GPIO
    sim_hardware.install()

simulation installed: pymycobot, RPi.GPIO and time.sleep are stand-ins, nothing will move


### The simulation (laptop only)
In the first cell, `sim_hardware.install()` (file `solutions/sim_hardware.py`) puts three stand-ins in place:
- a module `pymycobot.mycobot` whose `MyCobot` has the same constructor and `send_coords(coords, speed, mode)` signature as the real one,
  but only prints what it is asked to do;
- a module `RPi.GPIO` with `setmode`, `setup`, `output`, `cleanup`, printing every pin change;
- a replacement for `time.sleep` that adds the requested time to a **simulated clock** and waits only 1/100 of it.

**How it works**
- Because the stand-ins are put into `sys.modules` under the real names, the template's own `import` lines pick them up
  without any change. On the Pi `install()` is simply not called.
- The log lines read `[sim   1.50 s] send_coords [...]`: the simulated time since the start of the **current cell**
  (an IPython `pre_run_cell` hook resets the clock to 0 for every cell). So the printed times are what the real sequence would take.
- Only the **main thread** is fast-forwarded (`threading.current_thread() is threading.main_thread()`): Jupyter's own helper threads
  also call `time.sleep`, and counting theirs would falsify the clock.
- `sim_hardware.call_at(t, fn)` runs `fn` when the simulated clock passes `t`: used in Task 23 to simulate a key press.


---
## Setup and robot connection
## Task 1: Connect to the myCobot 280

In [2]:
from pymycobot.mycobot import MyCobot

serial_port = '/dev/serial0'   # the serial device used by the robot (as in motion_node.py)
baud_rate   = 1000000          # 1 Mbaud

mc = MyCobot(serial_port, baud_rate)
time.sleep(0.2)
print('MyCobot connected on', serial_port, 'at', baud_rate, 'baud')

[sim   0.00 s] connect /dev/serial0 1000000
MyCobot connected on /dev/serial0 at 1000000 baud


**How it works**
- `pymycobot` is the vendor's Python library: `MyCobot(port, baudrate)` opens the serial port and speaks the vendor protocol.
  Unlike Labs 2–7, pick and place uses **this library directly**, not `mycobot_control` and not `serial_iface.py`.
- Values from `motion_node.py`: `MyCobot("/dev/serial0", 1000000)`. (The library's own default baud rate is 115200, so the argument matters.)
- **Only one program may own the port**: close the Lab 7 serial notebook and stop `mycobot_control` first, otherwise
  the constructor fails or the two programs garble each other's frames.
- `time.sleep(0.2)`: a short pause so the connection is ready before the first command.
- In simulation the `from pymycobot.mycobot import MyCobot` line finds the stand-in installed in the setup cell; on the Pi it imports the real library. The cell itself is the template's.

---
## Workspace geometry and constants
## Task 2: Constants from `motion_node.py`

In [3]:
# Tool geometry
NOZZLE_LENGTH_MM = 68.0   # suction nozzle length (TCP offset)
HOVER_MARGIN     = 70.0   # clearance above pick height

# Motion timing
MOVE_SPEED   = 50
MOTION_SLEEP = 1.5   # seconds between moves

# Drop height
DROP_Z_MM = 60.0   # Z height for the drop position above the table

print(f'Nozzle: {NOZZLE_LENGTH_MM} mm, Hover margin: {HOVER_MARGIN} mm')
print(f'Speed: {MOVE_SPEED}, Sleep: {MOTION_SLEEP} s, Drop Z: {DROP_Z_MM} mm')

Nozzle: 68.0 mm, Hover margin: 70.0 mm
Speed: 50, Sleep: 1.5 s, Drop Z: 60.0 mm


**How it works**
- `NOZZLE_LENGTH_MM = 68`: the robot's Cartesian target is the **flange/tool reference** of the arm; the suction cup tip is 68 mm further along
  the tool axis. To put the tip at height *z* the flange must be at *z + 68*. This is the tool offset (TCP) of Lab 5 as a
  single number, applied by hand in the heights of Task 10.
- `HOVER_MARGIN = 70`: safety clearance above the pick height, so the arm approaches from above and does not hit the cube.
- `MOVE_SPEED = 50` (1–100) and `MOTION_SLEEP = 1.5` s: `send_coords` returns immediately (the robot moves on its own), so the code
  **waits a fixed time** for the move to finish. That is simple but not robust: a slow move that takes longer than 1.5 s would
  be interrupted by the next command. (`mc.is_moving()` or `is_in_position()` could be polled instead.)
- Constants in CAPITALS are a Python naming convention: "do not change while running".

---
## Task 3: Home and intermediate positions

In [4]:
HOME_COORDS  = [51.3, -63.3, 412.67, -91.75, -0.63, -89.98]
INTER_COORDS = [156.4, -17.2, 247.8, 179.27, 0.21, -1.4]

HOME_SPEED = 40
HOME_DELAY = 4.0   # seconds to wait after each home move

print('Home:', HOME_COORDS)
print('Intermediate:', INTER_COORDS)

Home: [51.3, -63.3, 412.67, -91.75, -0.63, -89.98]
Intermediate: [156.4, -17.2, 247.8, 179.27, 0.21, -1.4]


**How it works**
- A Cartesian pose is `[x, y, z, rx, ry, rz]`: position in **mm**, orientation in **degrees** (the robot API's convention).
- **Home** (z = 413 mm) is the arm stretched up with the camera looking at the table: the pose in which the vision system sees the
  workspace and the arm is out of the way. Its orientation `[-91.75, -0.63, -89.98]` is the "tool horizontal" pose of the
  home posture (compare `q = 0`, Lab 2).
- **Intermediate** waypoint (x = 156, z = 248, orientation ≈ `[180, 0, 0]`): a pose above the workspace with the tool already pointing
  down like in the pick sequence. Going through it turns the wrist gradually instead of swinging it from "pointing down" to "pointing
  sideways" in one move.
- `HOME_DELAY = 4 s`: home moves are long, so they wait longer than normal moves (`MOTION_SLEEP` = 1.5 s).

---
## Task 4: Workspace bounds

In [5]:
WORKSPACE_X_MIN = 75.0    # mm
WORKSPACE_X_MAX = 225.0   # mm
WORKSPACE_Y_MIN = -75.0   # mm
WORKSPACE_Y_MAX = 75.0    # mm

print(f'X: [{WORKSPACE_X_MIN}, {WORKSPACE_X_MAX}] mm')
print(f'Y: [{WORKSPACE_Y_MIN}, {WORKSPACE_Y_MAX}] mm')

X: [75.0, 225.0] mm
Y: [-75.0, 75.0] mm


**How it works**
- A **rectangle** in the base-frame XY plane, 150 mm × 150 mm, in front of the robot. Detections outside it are ignored.
- **Why:** the vision system only sees part of the table reliably; targets too close to the base or too far are unreachable
  or unsafe (collisions with bins, singular poses); and a wrong detection (reflection, mis-scaled depth) far outside the table
  must not turn into a robot motion (PDF question 8).

---
## Task 5: Colour-specific bins

In [6]:
BIN_COORDS = {
    "red":    [132.2, -147.9],
    "yellow": [245.8, -150.1],
    "green":  [115.8,  177.3],
    "blue":   [ -6.9,  173.2],
    "cyan":   [ -6.9,  173.2],
}

for color, coords in BIN_COORDS.items():
    print(f'{color:8s} → x={coords[0]:7.1f}, y={coords[1]:7.1f} mm')

red      → x=  132.2, y= -147.9 mm
yellow   → x=  245.8, y= -150.1 mm
green    → x=  115.8, y=  177.3 mm
blue     → x=   -6.9, y=  173.2 mm
cyan     → x=   -6.9, y=  173.2 mm


**How it works**
- A **dictionary** maps a colour name (string key) to the `[x, y]` position of its bin. Lookup: `BIN_COORDS["red"]` → `[132.2, -147.9]`.
  `.items()` gives (key, value) pairs for the loop.
- Only x and y are stored; the drop height is computed separately (Task 10).
- The bins lie **outside** the pick workspace (|y| > 147 mm, or x = 245 mm): the robot lifts the cube and moves sideways.
  Note that `blue` and `cyan` share one bin, and `blue` never occurs in the priority list of Task 14 (the vision model only has four classes:
  red, green, yellow, cyan).
- **Format spec in the f-string:** `{color:8s}` pads the text to 8 characters, `{coords[0]:7.1f}` prints a float with one decimal in
  7 characters. `→` is just a Unicode character in the text.

---
## Vacuum gripper control via GPIO
## Task 6: Set up the GPIO pins

In [7]:
import RPi.GPIO as GPIO

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)

# Configure pins
GPIO.setup(20, GPIO.OUT)   # the vacuum-motor control pin
GPIO.setup(21, GPIO.OUT)   # the release-valve control pin

# Initial state: both HIGH (off)
GPIO.output(20, 1)   # inactive state
GPIO.output(21, 1)   # inactive state

print('GPIO pins 20 and 21 configured and set HIGH (pump off)')

[sim   0.00 s] GPIO mode BCM
[sim   0.00 s] GPIO setup pin 20 OUT
[sim   0.00 s] GPIO setup pin 21 OUT
[sim   0.00 s] GPIO pin 20 <- 1
[sim   0.00 s] GPIO pin 21 <- 1
GPIO pins 20 and 21 configured and set HIGH (pump off)


**How it works**
- `RPi.GPIO` controls the Raspberry Pi's pins. `GPIO.setmode(GPIO.BCM)` selects **BCM numbering** (the chip's GPIO numbers, e.g. GPIO 20),
  not the physical pin numbers on the header. `setup(pin, GPIO.OUT)` makes a pin an output; `output(pin, 1)` sets it HIGH (3.3 V), `0` LOW (0 V).
- **Active-low:** the driver electronics switch *on* when the pin is LOW. So HIGH = off. The **initial state is HIGH/HIGH** so that nothing
  runs at start-up or after a crash: the safe state has to be the default.
- `GPIO.setwarnings(False)` silences the "channel already in use" warning when a program is restarted.
- This library only exists on the Pi (`import RPi.GPIO` fails on a laptop): the simulation installs a stand-in module.

---
## Task 7: `pump_on` and `pump_off`

In [8]:
def pump_on():
    """Activate suction: set both pins LOW."""
    GPIO.output(20, 0)   # vacuum motor ON (active-low)
    GPIO.output(21, 0)   # release valve CLOSED (active-low)
    print('Pump ON')

def pump_off():
    """Deactivate suction and release object."""
    GPIO.output(20, 1)   # motor off
    time.sleep(0.3)      # brief pause: let the pump stop
    GPIO.output(21, 1)   # release valve OPEN: air flows in, vacuum breaks
    time.sleep(2.0)      # wait long enough for the object to detach
    print('Pump OFF')

**How it works**
- `pump_on`: motor pin LOW (pump runs) **and** valve pin LOW (valve closed, so the air is drawn through the cup and a vacuum builds).
- `pump_off`: first the motor is switched off; after 0.3 s the valve pin goes HIGH (valve open), letting air flow into the
  system and destroying the vacuum; then a 2 s wait.
- **Question: why the 2.0 s delay after opening the release valve?** Because releasing is not instantaneous: the vacuum in the tube and cup has to
  decay through the valve, the suction force has to fall below the weight of the cube, and the cube has to actually fall off the cup
  (and stop bouncing) into the bin. If the arm moved away too early it would drag the cube along, or the cube would fall
  somewhere else. The sleep guarantees "released" before the next motion. (The 0.3 s before it lets the pump stop first, so that
  venting does not fight a running pump.)
- **Why two pins?** They are two independent actuators with different roles: the *motor* creates vacuum, the *valve* connects the
  cup to the air. With a single pin, switching the motor off would leave the vacuum in the line to decay slowly through leaks:
  the release would be slow and unreliable. Two pins allow *hold* (motor on, valve closed), *vent* (motor off, valve open),
  and a defined off state (both HIGH) (PDF question 7).

**Check with the simulation log:**

```python
pump_on()
time.sleep(1.0)
pump_off()
```

Output:
```text
[sim   0.00 s] GPIO pin 20 <- 0
[sim   0.00 s] GPIO pin 21 <- 0
Pump ON
[sim   1.00 s] GPIO pin 20 <- 1
[sim   1.30 s] GPIO pin 21 <- 1
Pump OFF
```


- Reading the log: both pins LOW at t = 0 (suction), then at t = 1.0 s motor HIGH, at 1.3 s the valve HIGH, and the function returns
  at 3.3 s. Nothing else moves during those 2.3 s.

---
## Task 8: Test the pump

In [10]:
enable_pump_test = SIMULATED   # real hardware: a person sets True to test with a small object held at the nozzle

if enable_pump_test:
    pump_on()
    time.sleep(3.0)
    pump_off()
    print('Pump test complete')
else:
    print('Pump test disabled. Set enable_pump_test = True to run.')

[sim   0.00 s] GPIO pin 20 <- 0
[sim   0.00 s] GPIO pin 21 <- 0
Pump ON
[sim   3.00 s] GPIO pin 20 <- 1
[sim   3.30 s] GPIO pin 21 <- 1
Pump OFF
Pump test complete


**How it works:** a **guard flag**: the hardware is only touched if the flag is `True`. On the real robot this cell must not run
by accident (e.g. "Run All"): the flag is the deliberate human decision. Hold a light object at the nozzle to verify suction, and
the cube should fall off during `pump_off`.

---
## Cartesian motion fundamentals
## Task 9: Go home via the intermediate waypoint

In [11]:
def go_home(mc):
    """Return to home via intermediate waypoint."""
    mc.send_coords(INTER_COORDS, HOME_SPEED, 0)   # first the intermediate waypoint ...
    time.sleep(HOME_DELAY)
    mc.send_coords(HOME_COORDS, HOME_SPEED, 0)    # ... then the home pose
    time.sleep(HOME_DELAY)
    print('Robot at home position')

# Execute
enable_motion = SIMULATED   # real robot: a person sets True after clearing the workspace

if enable_motion:
    go_home(mc)
else:
    print('Motion disabled. Set enable_motion = True to move the robot.')

[sim   0.00 s] send_coords [156.4, -17.2, 247.8, 179.3, 0.2, -1.4] speed 40 mode 0
[sim   4.00 s] send_coords [51.3, -63.3, 412.7, -91.8, -0.6, -90.0] speed 40 mode 0


Robot at home position


**How it works**
- `mc.send_coords([x, y, z, rx, ry, rz], speed, mode)`: move to a Cartesian pose. Position in mm, orientation in degrees, speed 1–100.
  **`mode`: 0 = angular, 1 = linear.** In angular mode the joints move simultaneously to the target (fast, the tool path is curved); in linear mode
  the tool goes in a straight line (slower, may fail near singularities). Mode 0 is used here. If the argument is left out,
  pymycobot sends the command and waits for the robot's reply instead.
- The function returns immediately; the `time.sleep` is the "wait until done".
- **Why via INTER_COORDS:** see the reflection (Task 28, question 2). The direct path from a "tool down" pose to home would swing the
  wrist through the workspace, near the bins and cubes.
- `def go_home(mc):` takes the robot object as an argument (instead of using the global), so it could be tested with a fake one.

---
## Task 10: Pick, hover and drop heights

In [12]:
# Simulated object height (mm)
z_obj = 40.0

z_pick  = z_obj + NOZZLE_LENGTH_MM               # end-effector pick height: nozzle tip touches the object
z_hover = z_pick + HOVER_MARGIN                  # safe clearance above the pick point
z_drop  = DROP_Z_MM + NOZZLE_LENGTH_MM           # end-effector drop height above the bin

print(f'Object Z:  {z_obj:.1f} mm')
print(f'Pick Z:    {z_pick:.1f} mm   (nozzle tip at object surface)')
print(f'Hover Z:   {z_hover:.1f} mm  (safe clearance above object)')
print(f'Drop Z:    {z_drop:.1f} mm   (release height above bin)')

Object Z:  40.0 mm
Pick Z:    108.0 mm   (nozzle tip at object surface)
Hover Z:   178.0 mm  (safe clearance above object)
Drop Z:    128.0 mm   (release height above bin)


**How it works**
- All three are heights of the **flange/tool reference point** that `send_coords` positions:
  - pick: `z_obj + 68` = 108 mm (the nozzle tip, 68 mm below it, is at 40 mm, the object height);
  - hover: `z_pick + 70` = 178 mm;
  - drop: `60 + 68` = 128 mm (fixed: the tip is 60 mm above the table at the bin, independent of the cube).
- The hover height depends on the object (higher stack, higher hover): also the way to the bin is travelled at the hover
  height *of the object*, so a cube being carried clears the cubes it is passing over.
- `f'{value:.1f}'` prints one decimal.

---
## Task 11: A single Cartesian move

In [13]:
# A safe test target within the workspace
test_target = [150.0, 0.0, 250.0, 180, 0, 0]

enable_motion = SIMULATED   # real robot: a person sets True

if enable_motion:
    mc.send_coords(test_target, MOVE_SPEED)
    time.sleep(MOTION_SLEEP)
    print('Moved to test target:', test_target)

    # Return home
    go_home(mc)
else:
    print('Motion disabled. Set enable_motion = True to move the robot.')

[sim   0.00 s] send_coords [150.0, 0.0, 250.0, 180.0, 0.0, 0.0] speed 50 mode None


Moved to test target: [150.0, 0.0, 250.0, 180, 0, 0]
[sim   1.50 s] send_coords [156.4, -17.2, 247.8, 179.3, 0.2, -1.4] speed 40 mode 0


[sim   5.50 s] send_coords [51.3, -63.3, 412.7, -91.8, -0.6, -90.0] speed 40 mode 0
Robot at home position


**How it works**
- `[150, 0, 250, 180, 0, 0]`: 150 mm in front of the base, in the middle (y = 0), 250 mm high, tool pointing **straight down**
  (roll 180° flips the tool z-axis from "up" to "down"). This is the orientation `[180, 0, 0]` used for the whole pick and place.
- `mc.send_coords(test_target, MOVE_SPEED)` without a mode. `time.sleep(MOTION_SLEEP)` gives it 1.5 s.
- Afterwards the robot returns home so it always ends in a known pose.
- Before running on the real robot: workspace clear, target inside the bounds of Task 4 (it is: x = 150, y = 0), and speed 50 is
  moderate but not slow. Start with a lower `MOVE_SPEED` if in doubt.

---
## Pick-and-place state machine
## Task 12: The eight-step sequence

In [14]:
def execute_pick_and_place(mc, obj_x, obj_y, obj_z, color):
    """
    Execute the full pick-and-place sequence.

    Parameters
    ----------
    mc : MyCobot
        Robot connection object.
    obj_x, obj_y, obj_z : float
        Object position in mm (robot base frame).
    color : str
        Detected colour string (key into BIN_COORDS).
    """
    # Compute heights
    pick_ee_z  = obj_z + NOZZLE_LENGTH_MM        # end-effector pick height
    hover_ee_z = pick_ee_z + HOVER_MARGIN        # hover height

    print(f'Pick Z: {pick_ee_z:.1f}, Hover Z: {hover_ee_z:.1f}')

    # Step 1: Hover above object
    mc.send_coords([obj_x, obj_y, hover_ee_z, 180, 0, 0], MOVE_SPEED)
    time.sleep(MOTION_SLEEP)

    # Step 2: Pump ON
    pump_on()

    # Step 3: Descend to pick height
    mc.send_coords([obj_x, obj_y, pick_ee_z, 180, 0, 0], MOVE_SPEED)
    time.sleep(MOTION_SLEEP)

    # Step 4: Lift to hover height
    mc.send_coords([obj_x, obj_y, hover_ee_z, 180, 0, 0], MOVE_SPEED)
    time.sleep(MOTION_SLEEP)

    # Step 5: Move to colour-specific bin
    bx, by = BIN_COORDS[color]                                   # bin coordinates for this colour
    mc.send_coords([bx, by, hover_ee_z, 180, 0, 0], MOVE_SPEED)  # above the selected bin
    time.sleep(MOTION_SLEEP)

    # Step 6: Descend to drop height
    drop_ee_z = DROP_Z_MM + NOZZLE_LENGTH_MM                     # end-effector drop height
    mc.send_coords([bx, by, drop_ee_z, 180, 0, 0], MOVE_SPEED)
    time.sleep(MOTION_SLEEP)

    # Step 7: Release object
    pump_off()

    # Step 8: Retreat and return home
    mc.send_coords([bx, by, hover_ee_z, 180, 0, 0], MOVE_SPEED)  # lift above the bin again
    time.sleep(MOTION_SLEEP)
    go_home(mc)

    print(f'Pick-and-place complete for {color} object')

**How it works, step by step** (for the object at (150, 0, 40) mm, colour red)

| step | action | robot pose `[x, y, z, 180, 0, 0]` | pump |
|---|---|---|---|
| 1 | hover above the object | (150, 0, 178) | off |
| 2 | pump on (before descending, so suction is established when the nozzle arrives) | | **on** |
| 3 | descend to the cube | (150, 0, 108) | on |
| 4 | lift the cube | (150, 0, 178) | on |
| 5 | move sideways over the bin | (132.2, −147.9, 178) | on |
| 6 | descend to the drop height | (132.2, −147.9, 128) | on |
| 7 | release (motor off, vent, wait) | | **off** |
| 8 | retreat up, then home via the waypoint | (132.2, −147.9, 178) → INTER → HOME | off |

- `bx, by = BIN_COORDS[color]`: **tuple unpacking** of the two-element list. A colour that is not in the dictionary raises a `KeyError`
  (the node checks first and calls `pump_off()`: see Task 25).
- Every move is followed by `time.sleep(MOTION_SLEEP)`: the sequence is **open loop**, nothing verifies that the arm has arrived
  or that the cube is really held.
- The list `[obj_x, obj_y, hover_ee_z, 180, 0, 0]` keeps the tool pointing down throughout: only x, y, z change.

---
## Task 13: Test the sequence

In [15]:
# Simulated detection (values in mm, as if already converted from metres)
test_obj_x = 150.0
test_obj_y = 0.0
test_obj_z = 40.0
test_color = 'red'

# Verify workspace bounds
in_bounds = (WORKSPACE_X_MIN <= test_obj_x <= WORKSPACE_X_MAX and
             WORKSPACE_Y_MIN <= test_obj_y <= WORKSPACE_Y_MAX)
print(f'Target in workspace: {in_bounds}')

enable_motion = SIMULATED   # real robot: a person sets True after placing the cube and clearing the workspace

if enable_motion and in_bounds:
    execute_pick_and_place(mc, test_obj_x, test_obj_y, test_obj_z, test_color)
else:
    print('Motion disabled or target out of bounds.')

Target in workspace: True
Pick Z: 108.0, Hover Z: 178.0
[sim   0.00 s] send_coords [150.0, 0.0, 178.0, 180.0, 0.0, 0.0] speed 50 mode None


[sim   1.50 s] GPIO pin 20 <- 0
[sim   1.50 s] GPIO pin 21 <- 0
Pump ON
[sim   1.50 s] send_coords [150.0, 0.0, 108.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   3.00 s] send_coords [150.0, 0.0, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   4.50 s] send_coords [132.2, -147.9, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   6.00 s] send_coords [132.2, -147.9, 128.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   7.50 s] GPIO pin 20 <- 1
[sim   7.80 s] GPIO pin 21 <- 1


Pump OFF
[sim   9.80 s] send_coords [132.2, -147.9, 178.0, 180.0, 0.0, 0.0] speed 50 mode None


[sim  11.30 s] send_coords [156.4, -17.2, 247.8, 179.3, 0.2, -1.4] speed 40 mode 0


[sim  15.30 s] send_coords [51.3, -63.3, 412.7, -91.8, -0.6, -90.0] speed 40 mode 0
Robot at home position
Pick-and-place complete for red object


**How it works**
- **Chained comparison** `A <= x <= B` is Python's readable way to write "x between A and B"; `and` combines the two axes.
- The `if enable_motion and in_bounds:` needs **both**: the flag (human decision) and the plausibility check.
- **Reading the log (simulation):** six plain moves (1.5 s each = 9 s), the release (0.3 s + 2 s), and two home moves (4 s each): the last home move is sent at 15.3 s and the
  cycle is finished after 19.3 s on the real robot. The sleeps dominate; the robot itself is idle for most of that time.

---
## Colour-based bin sorting and priority logic
## Task 14: Priority selection

In [16]:
COLOR_PRIORITY = {
    "red":    0,
    "blue": 1,
    "yellow":  2,
    #"cyan":   3,
}

def is_inside_workspace(x_m, y_m):
    """Check if a detection (in metres) is inside the workspace."""
    x_mm = x_m * 1000.0
    y_mm = y_m * 1000.0
    return (WORKSPACE_X_MIN <= x_mm <= WORKSPACE_X_MAX and
            WORKSPACE_Y_MIN <= y_mm <= WORKSPACE_Y_MAX)   # apply the configured workspace limits

def choose_target(detections):
    """
    Select the highest-priority target from a list of detections.

    Each detection is a dict: {'x': float, 'y': float, 'z': float, 'color': str}
    where x, y, z are in metres.

    Returns the selected detection dict, or None.
    """
    valid = [
        d for d in detections
        if d['color'] in COLOR_PRIORITY and is_inside_workspace(d['x'], d['y'])
    ]   # keep only detections with a supported priority and a valid workspace position

    if not valid:
        return None

    # Sort: highest z first (descending), then colour priority (ascending)
    valid.sort(key=lambda d: (-d['z'], COLOR_PRIORITY[d['color']]))   # stack height first, then colour priority
    return valid[0]

**How it works**
- **List comprehension with a condition:** `[d for d in detections if cond]` builds the list of detections that satisfy both conditions: the colour has
  a priority entry (`d['color'] in COLOR_PRIORITY` tests dictionary **keys**) and the position is inside the workspace.
- `valid.sort(key=...)` sorts the list **in place**. The key is a function; `lambda d: (-d['z'], COLOR_PRIORITY[d['color']])` returns a
  **tuple**, and Python compares tuples element by element: first by `-z` (ascending = highest z first), and only for equal `z` by the
  colour priority (0 = most urgent).
- **Why negate `z`?** Sorting ascending puts the smallest key first; the minus sign turns "largest z first" into "smallest key first".
- `valid[0]` is then the best candidate.
- **Why higher stacks first?** A cube on top of another one must be removed first; the lower one is only reachable afterwards.
- Tuples are compared exactly, so two cubes at the same stack height but with z = 0.0400001 vs 0.04 (float noise) would not tie. In the real
  system z takes discrete values (`Z_absolute = Z_ONE - level * h`) so ties are exact.
- Exactly as in `choose_highest_priority_target` of `motion_node.py` (there the detections are `DetectedObject` messages and the
  fields are attributes `d.z`, `d.color`).

---
## Task 15: Test the priority selection

In [17]:
# Simulated detection buffer (values in metres)
detections = [
    {'x': 0.150, 'y': 0.000, 'z': 0.040, 'color': 'green'},
    {'x': 0.120, 'y': 0.030, 'z': 0.040, 'color': 'red'},
    {'x': 0.200, 'y': -0.050, 'z': 0.080, 'color': 'yellow'},   # stacked: higher z
    {'x': 0.300, 'y': 0.000, 'z': 0.040, 'color': 'cyan'},      # outside workspace
]

target = choose_target(detections)

if target is not None:
    print(f'Selected: {target["color"]} at '
          f'x={target["x"]:.3f}, y={target["y"]:.3f}, z={target["z"]:.3f} m')
else:
    print('No valid target found')

# Expected: yellow at z=0.080 (highest stack), since it has the highest z
# If two had the same z, red (priority 0) would be chosen over green (priority 2)

Selected: yellow at x=0.200, y=-0.050, z=0.080 m


**How it works**
- The cyan cube is at x = 300 mm > 225 mm: **filtered out** (outside the workspace) before sorting.
- Of the three valid detections, yellow has the highest z (0.08 = stacked) and wins, although red has the better colour priority.
- `dict(d, z=0.040)` copies dictionary `d` and overrides `z`: with all heights equal the tie-break by priority picks red
  (priority 0), as the last comment says.

**Example: the tie-break (same stack height everywhere)**

```python
same_height = [dict(d, z=0.040) for d in detections]
print('same heights ->', choose_target(same_height)['color'])
```

Output:
```text
same heights -> red
```


---
## Workspace validation
## Task 16: Test the bounds check

In [19]:
test_positions = [
    (0.150, 0.000, 'centre of workspace'),
    (0.075, -0.075, 'corner — just inside'),
    (0.050, 0.000, 'too far left (x too small)'),
    (0.250, 0.000, 'too far right (x too large)'),
    (0.150, 0.100, 'too far forward (y too large)'),
]

for x_m, y_m, label in test_positions:
    result = is_inside_workspace(x_m, y_m)
    status = 'INSIDE' if result else 'OUTSIDE'
    print(f'  ({x_m:.3f}, {y_m:.3f}) m → {x_m*1000:.0f}, {y_m*1000:.0f} mm → {status:7s}  ({label})')

  (0.150, 0.000) m → 150, 0 mm → INSIDE   (centre of workspace)
  (0.075, -0.075) m → 75, -75 mm → INSIDE   (corner — just inside)
  (0.050, 0.000) m → 50, 0 mm → OUTSIDE  (too far left (x too small))
  (0.250, 0.000) m → 250, 0 mm → OUTSIDE  (too far right (x too large))
  (0.150, 0.100) m → 150, 100 mm → OUTSIDE  (too far forward (y too large))


**How it works**
- The function receives **metres** (as published by the vision node) and converts to **mm** for the comparison with the limits
  (given in mm). Forgetting the conversion would compare 0.15 with 75…225 and reject every detection.
- `'INSIDE' if result else 'OUTSIDE'`: a conditional expression; `{status:7s}` pads the text.
- The corner (0.075, −0.075) is exactly on the boundary; because the comparison uses `<=` it is **inside**. On a boundary, floating-point
  rounding could decide (`0.075 * 1000` might come out as 74.99999999999999). Here it prints exactly 75.0 and −75.0, so the corner passes;
  in general one uses a small tolerance or avoids objects at the boundary.

**Example: the boundary corner in floating point**

```python
print('0.075 * 1000 =', repr(0.075 * 1000), '  -0.075 * 1000 =', repr(-0.075 * 1000))
```

Output:
```text
0.075 * 1000 = 75.0   -0.075 * 1000 = -75.0
```


---
## Camera-to-robot base coordinate transform
## Task 17: The 4×4 matrix `T_base_cam`

In [21]:
R_base_cam = np.array([
    [0, 1, 0],     # base x  =  camera y
    [1, 0, 0],     # base y  =  camera x
    [0, 0, -1],    # base z  = -camera z
], dtype=np.float32)

t_base_cam = np.array([0.170, 0.0, 0.39], dtype=np.float32)   # camera origin in the base frame (m)

T_base_cam = np.eye(4, dtype=np.float32)
T_base_cam[:3, :3] = R_base_cam    # rotation component
T_base_cam[:3, 3]  = t_base_cam    # translation component

print('T_base_cam:')
print(T_base_cam)

T_base_cam:
[[ 0.    1.    0.    0.17]
 [ 1.    0.    0.    0.  ]
 [ 0.    0.   -1.    0.39]
 [ 0.    0.    0.    1.  ]]


**How it works** (PDF question 2: the matrix is)
$$ {}^{base}T_{cam} = \begin{bmatrix} 0 & 1 & 0 & 0.170\\ 1 & 0 & 0 & 0.000\\ 0 & 0 & -1 & 0.390\\ 0&0&0&1\end{bmatrix} $$
- **Reading the columns of R** (Lab 1: column *i* = axis *i* of the rotated frame in base coordinates): column 0 = `[0, 1, 0]`: the
  camera x-axis points along the base **y**; column 1 = `[1, 0, 0]`: the camera y-axis points along the base **x**; column 2 =
  `[0, 0, -1]`: the camera z-axis (the viewing direction) points **down**, along negative base z. The camera looks down at the table.
- `t = [0.170, 0, 0.39]`: the camera sits 17 cm in front of the base and 39 cm above the base plane.
- `dtype=np.float32`: the vision node works in 32-bit floats (matches OpenCV and the ROS message type `float32`).
- The check: det = +1 and `R.T @ R = I`, so it is a proper rotation (not a mirror image), even though it looks like a "swap and flip":
  swapping x and y is a reflection, flipping z is another, and two reflections make a rotation (by 180° about the axis (1, 1, 0)/√2).

**Example: is it a proper rotation?**

```python
print('det(R) =', np.linalg.det(R_base_cam), '  R^T R = I:', np.allclose(R_base_cam.T @ R_base_cam, np.eye(3)))
```

Output:
```text
det(R) = 1.0   R^T R = I: True
```


---
## Task 18: Apply the transform to a simulated detection

In [23]:
# Simulated camera-frame object pose
R_cam_obj = np.eye(3, dtype=np.float32)   # identity for simplicity
t_cam_obj = np.array([0.02, -0.01, 0.36], dtype=np.float32)

T_cam_obj = np.eye(4, dtype=np.float32)
T_cam_obj[:3, :3] = R_cam_obj
T_cam_obj[:3, 3]  = t_cam_obj

# Transform to base frame
T_base_obj = T_base_cam @ T_cam_obj

object_pos_base = T_base_obj[:3, 3]
print(f'Object position in base frame: x={object_pos_base[0]:.4f}, '
      f'y={object_pos_base[1]:.4f}, z={object_pos_base[2]:.4f} m')


Object position in base frame: x=0.1600, y=0.0200, z=0.0300 m


**How it works**
- ${}^{base}T_{obj} = {}^{base}T_{cam}\,{}^{cam}T_{obj}$: base ← camera ← object, the chain rule of Lab 1.
- Position: $p_{base} = R\,p_{cam} + t = R[0.02, -0.01, 0.36]^T + [0.17, 0, 0.39]^T = [-0.01, 0.02, -0.36] + [0.17, 0, 0.39] = [0.16, 0.02, 0.03]$ m.
  Read it: the object is 2 cm to the **right in the image (camera x)** = **+2 cm along the base y**; 1 cm "up" in the image (camera y = −1 cm) = 1 cm **closer to the base** in x
  (x = 0.17 − 0.01); and 0.36 m in front of the camera lens along its viewing axis = 39 − 36 = **3 cm above the table**.
- So `Z_ONE_CUBE = 0.36` is really **the depth of the top of one cube measured from the camera**, not a base-frame height. The PDF calls it "base Z"; what enters the transform is a
  camera-frame depth. In the base frame this cube is at z = 3 cm. (`T_cam_obj[:3, 3] = [tvec[0][0], tvec[1][0], Z_absolute]` in the vision node.)
- The `@` operator and the by-hand computation agree (as in Lab 1 tasks 5/10).

**Example: the same by hand (rotate, then add the camera position)**

```python
print('by hand:', R_base_cam @ t_cam_obj + t_base_cam)
```

Output:
```text
by hand: [0.16       0.02       0.02999997]
```


---
## Task 19: Z-scale calibration
$$ f_{scale} = \frac{Z_{one}}{z_{ref}},\qquad z_{corrected} = z_{raw}\cdot f_{scale} $$

In [25]:
Z_ONE_CUBE = 0.36       # true depth for one cube (metres)
CUBE_HEIGHT = 0.04      # height of one cube (metres)

# Example calibration data
z_ref_raw = 0.385       # average raw PnP depth from 5 samples

f_scale = Z_ONE_CUBE / z_ref_raw        # depth-correction scale factor

# A new raw measurement
z_raw_new = 0.370
z_corrected = z_raw_new * f_scale       # apply the calibration scale to the raw depth

print(f'Scale factor:    {f_scale:.4f}')
print(f'Raw depth:       {z_raw_new:.4f} m')
print(f'Corrected depth: {z_corrected:.4f} m')

Scale factor:    0.9351
Raw depth:       0.3700 m
Corrected depth: 0.3460 m


**How it works**
- **Why a scale factor?** `solvePnP` estimates the distance from a small object seen by a camera; errors in the camera intrinsics, the
  keypoint positions and the assumed cube size (4 cm) mostly scale **all depths by the same factor**. One cube at a *known* depth (0.36 m) tells the factor:
  the average of five raw depths (0.385 m) should have been 0.36 m, so every raw depth is multiplied by $0.36/0.385 = 0.935$.
- Averaging five frames reduces noise in the reference measurement.
- Correcting the new raw value 0.370 m gives 0.346 m. Which is 14 mm smaller than the one-cube depth, less than half a cube (20 mm): Task 20 will call it level 0.
- In the real vision node the factor is measured once in calibration mode (key `r` recalibrates) and stored in `z_scale_calibration.json`.

---
## Task 20: Stack level from the corrected depth
$$ \Delta z = z_{corrected} - Z_{one},\qquad \ell = \max\big(0,\ \operatorname{round}(-\Delta z/h_{cube})\big),\qquad Z_{abs} = Z_{one} - \ell\,h_{cube} $$

In [26]:
def compute_stack_level(z_corrected):
    delta_z = z_corrected - Z_ONE_CUBE                        # depth difference from the one-cube reference
    level = max(0, round(-delta_z / CUBE_HEIGHT))             # non-negative stack level
    Z_absolute = Z_ONE_CUBE - level * CUBE_HEIGHT             # depth used for that stack level
    return level, Z_absolute

# Test
for z_c in [0.36, 0.32, 0.28]:
    lvl, z_abs = compute_stack_level(z_c)
    print(f'z_corrected={z_c:.2f} m → level={lvl}, Z_absolute={z_abs:.3f} m')

z_corrected=0.36 m → level=0, Z_absolute=0.360 m
z_corrected=0.32 m → level=1, Z_absolute=0.320 m
z_corrected=0.28 m → level=2, Z_absolute=0.280 m


**How it works**
- A cube on top of another one is 4 cm closer to the camera, so its top face is at a **smaller** depth: $\Delta z = -0.04$ per level.
  Dividing by the cube height gives the level as a fractional number; `round(...)` snaps it to the nearest integer, which makes the
  estimate robust against noise of up to ±2 cm.
- `max(0, ...)`: a depth *larger* than the reference (noise, uneven table) would give a negative level, which is meaningless: clamp to 0.
- Results: 0.36 → level 0 (single cube); 0.32 → level 1 (stack of two); 0.28 → level 2.
- **`Z_absolute` is then the *snapped* depth** (0.36, 0.32, 0.28), independent of the noise in the measurement: the stack height is
  quantised on purpose. In the base frame it becomes z = 0.39 − Z_abs = 0.03, 0.07, 0.11 m (Task 18).
- Note Python's `round` rounds halves to the **nearest even** integer (`round(0.5) == 0`, `round(1.5) == 2`, `round(2.5) == 2`), and floating-point noise decides at exactly half a cube:
  the example below shows level 1 for 0.340 m although Δz = −2 cm. Half a cube is a coin flip either way; what matters is that the tolerance around each level is about ±2 cm.

**Example: the tolerance around each level**

```python
for z_c in np.arange(0.345, 0.285, -0.005):
    lvl, z_abs = compute_stack_level(z_c)
    print(f'z_corrected = {z_c:.3f} m  ->  level {lvl}', '  (borderline)' if abs((0.36 - z_c) / 0.04 - round((0.36 - z_c) / 0.04)) > 0.45 else '')
```

Output:
```text
z_corrected = 0.345 m  ->  level 0 
z_corrected = 0.340 m  ->  level 1   (borderline)
z_corrected = 0.335 m  ->  level 1 
z_corrected = 0.330 m  ->  level 1 
z_corrected = 0.325 m  ->  level 1 
z_corrected = 0.320 m  ->  level 1 
z_corrected = 0.315 m  ->  level 1 
z_corrected = 0.310 m  ->  level 1 
z_corrected = 0.305 m  ->  level 1 
z_corrected = 0.300 m  ->  level 2   (borderline)
z_corrected = 0.295 m  ->  level 2 
z_corrected = 0.290 m  ->  level 2 
```


---
## Detection buffer and timing
## Task 21: Simulate the buffer

In [28]:
DETECTION_WAIT_TIME = 2.0   # seconds

# Simulated detection stream: (time_offset_s, detection_dict)
detection_stream = [
    (0.0,  {'x': 0.150, 'y': 0.000, 'z': 0.040, 'color': 'green'}),
    (0.5,  {'x': 0.120, 'y': 0.030, 'z': 0.040, 'color': 'red'}),
    (1.0,  {'x': 0.200, 'y': -0.050, 'z': 0.080, 'color': 'yellow'}),
    (1.8,  {'x': 0.180, 'y': 0.020, 'z': 0.040, 'color': 'cyan'}),
    (2.5,  {'x': 0.160, 'y': 0.010, 'z': 0.040, 'color': 'red'}),   # arrives after wait
]

# Collect detections within the wait window
buffer = [d for t, d in detection_stream if t < DETECTION_WAIT_TIME]

print(f'Buffered {len(buffer)} detections')
for d in buffer:
    print(f'  {d["color"]:8s}  z={d["z"]:.3f} m')

target = choose_target(buffer)
if target:
    print(f'\nSelected target: {target["color"]} at z={target["z"]:.3f} m')

Buffered 4 detections
  green     z=0.040 m
  red       z=0.040 m
  yellow    z=0.080 m
  cyan      z=0.040 m

Selected target: yellow at z=0.080 m


**How it works**
- The first detection (t = 0) starts the timer; everything with `t < 2.0` s is collected: 4 detections. The one at 2.5 s is too late.
- `[d for t, d in detection_stream if t < DETECTION_WAIT_TIME]`: iterates over the (time, detection) pairs, unpacks each, keeps the detection
  if the time is inside the window.
- `choose_target(buffer)` then picks yellow (stack, z = 0.08).
- **Why wait at all?** The vision runs on single frames; a cube can be missed in a frame, be partly hidden, or be seen a moment later
  (YOLO flicker). Choosing from the *first* frame could miss a stacked cube or a higher-priority colour that appears 0.5 s
  later. Two seconds of collected frames give a fuller picture of the scene (PDF question 9).
- **Detail from the real node:** the selection is triggered by the **next detection message after the window**, not by a timer; that message itself is not in the buffer. If no
  further message arrives, nothing happens until one does.

---
## Unit conversion: metres to millimetres
## Task 22: Convert detection coordinates

In [29]:
# Simulated DetectedObject values (metres)
det_x_m = 0.150
det_y_m = -0.030
det_z_m = 0.040

# Convert to mm
obj_x = det_x_m * 1000.0
obj_y = det_y_m * 1000.0
obj_z = det_z_m * 1000.0

print(f'Detection (m):  x={det_x_m:.3f}, y={det_y_m:.3f}, z={det_z_m:.3f}')
print(f'Converted (mm): x={obj_x:.1f}, y={obj_y:.1f}, z={obj_z:.1f}')

# Compute end-effector heights
pick_z  = obj_z + NOZZLE_LENGTH_MM             # end-effector pick height
hover_z = pick_z + HOVER_MARGIN                # safe hover height
drop_z  = DROP_Z_MM + NOZZLE_LENGTH_MM         # end-effector drop height

print(f'\nPick Z:  {pick_z:.1f} mm')
print(f'Hover Z: {hover_z:.1f} mm')
print(f'Drop Z:  {drop_z:.1f} mm')

Detection (m):  x=0.150, y=-0.030, z=0.040
Converted (mm): x=150.0, y=-30.0, z=40.0

Pick Z:  108.0 mm
Hover Z: 178.0 mm
Drop Z:  128.0 mm


**How it works:** `DetectedObject` uses **metres** (ROS convention, SI), the robot API uses **millimetres**: multiply by 1000 once, in one place
(`execute_pick_and_place` in the node does it). Mixing the two would send the robot to 0.15 mm or to 150 m. The
heights are the same formulas as in Task 10 (pick 108, hover 178, drop 128).

---
## Safety mechanisms
## Task 23: The stop flag

In [30]:
stop_flag = {"stop": False}

def check_stop():
    """Check the stop flag. If set, turn off the pump and return True."""
    if stop_flag["stop"]:
        print('[SAFETY] Motion stopped!')
        pump_off()          # release the object
        return True
    return False

# Test the logic without actuating real GPIO
stop_flag["stop"] = False
print('Stop check (no stop):', check_stop())

# For the True case, test the flag detection without calling pump_off on real hardware
stop_flag["stop"] = True
if stop_flag["stop"]:
    print('Stop check (stop requested): stop flag correctly detected')
    print('  (pump_off() would be called in a real sequence)')

# Reset for later use
stop_flag["stop"] = False

Stop check (no stop): False
Stop check (stop requested): stop flag correctly detected
  (pump_off() would be called in a real sequence)


**How it works**
- The flag lives in a **dictionary** (`{"stop": False}`), not in a plain variable: a dictionary is **mutable** and shared, so a second thread
  (the keyboard listener in the node, which sets `stop_flag["stop"] = True` when you press `s`) and the sequence code see the same object. A
  plain `stop = False` variable would be copied by value into the thread.
- `check_stop()` returns `True` after switching the pump off, so the caller can leave the sequence. Between motions, not
  during them: the check runs **before each `send_coords`**.
- **Improved sequence with stop checks** (as in the node: `if self.check_stop(): return` before every move):

```python
def execute_with_stop_checks(mc, obj_x, obj_y, obj_z, color):
    """Task 12 sequence with a stop check before every move (as motion_node.py)."""
    pick_ee_z = obj_z + NOZZLE_LENGTH_MM
    hover_ee_z = pick_ee_z + HOVER_MARGIN
    if color not in BIN_COORDS:                       # refuse BEFORE picking anything up
        print(f"Unknown colour '{color}': nothing done")
        return False
    bx, by = BIN_COORDS[color]
    drop_ee_z = DROP_Z_MM + NOZZLE_LENGTH_MM

    steps = [                                         # (label, pose) for the moves that need a stop check first
        ('hover', [obj_x, obj_y, hover_ee_z, 180, 0, 0]),
        ('descend', [obj_x, obj_y, pick_ee_z, 180, 0, 0]),
        ('lift', [obj_x, obj_y, hover_ee_z, 180, 0, 0]),
        ('to bin', [bx, by, hover_ee_z, 180, 0, 0]),
        ('lower', [bx, by, drop_ee_z, 180, 0, 0]),
    ]
    for label, pose in steps:
        if check_stop():
            return False
        mc.send_coords(pose, MOVE_SPEED)
        time.sleep(MOTION_SLEEP)
        if label == 'hover':
            pump_on()                                 # step 2: suction on after reaching the hover pose

    pump_off()                                        # step 7
    if check_stop():
        return False
    mc.send_coords([bx, by, hover_ee_z, 180, 0, 0], MOVE_SPEED)   # retreat
    time.sleep(MOTION_SLEEP)
    go_home(mc)
    print(f'Pick-and-place complete for {color} object')
    return True

# quick test in simulation: "the user presses s" at simulated t = 4.0 s
sim_hardware.call_at(4.0, lambda: stop_flag.update(stop=True))
print('completed:', execute_with_stop_checks(mc, 150.0, 0.0, 40.0, 'red'))
stop_flag['stop'] = False
```

Output:
```text
[sim   0.00 s] send_coords [150.0, 0.0, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   1.50 s] GPIO pin 20 <- 0
[sim   1.50 s] GPIO pin 21 <- 0
Pump ON
[sim   1.50 s] send_coords [150.0, 0.0, 108.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   3.00 s] send_coords [150.0, 0.0, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[SAFETY] Motion stopped!
[sim   4.50 s] GPIO pin 20 <- 1
[sim   4.80 s] GPIO pin 21 <- 1
Pump OFF
completed: False
```


- This is a separate function (`execute_with_stop_checks`), so the Task 12 version stays as the template has it.
- **`steps` list:** the five plain moves are a list of `(label, pose)` tuples processed by one loop, with a `check_stop()` before each. That is shorter than
  five copies of the same three lines and puts the safety check in **one** place, so it cannot be forgotten.
- The unknown-colour check happens **first**: in the template version `BIN_COORDS[color]` would raise a `KeyError` in the middle of the sequence, with the cube
  already in the gripper and the pump running.
- The test calls `sim_hardware.call_at(4.0, ...)`: when the simulated clock passes 4.0 s, the fast-forwarded `sleep` runs `stop_flag.update(stop=True)`, which stands in for a
  key press. (`lambda: ...` is a one-line anonymous function; `dict.update(stop=True)` sets the key.) Expected log: hover at 0 s, pump on at 1.5 s, descend, lift at 3.0 s; the flag appears during the sleep that ends at 4.5 s, so `check_stop()`
  in front of the next move ("to bin") releases the pump (0.3 s + 2 s) and the function returns `False`. No move after that.
- **A limit of this mechanism:** the check happens *between* moves. A move that has been sent already runs to its end (`send_coords` cannot be recalled). For a true emergency
  use the robot's own stop (`mc.stop()`/power) or the Lab 7 stop frame.

---
## Task 24: The safety mechanisms of the motion node
| # | mechanism | purpose and the failure it guards against |
|---|---|---|
| 1 | **`busy` flag** | Set while a pick-and-place runs; `object_callback` returns immediately while it is set. Prevents a **second sequence from being started** by a new detection (or by the idle timer's `go_home`) while the arm is already moving: two overlapping motion sequences would send contradictory commands, and could drop the cube. (In a single-threaded ROS executor callbacks cannot literally overlap, but the flag also protects the timer and any multi-threaded use.) |
| 2 | **`at_home` flag** | New detections are only accepted when the arm is at home. Guards against acting on **bad images**: with the arm in the field of view (or holding a cube) the camera sees the arm and moving objects, and any detection made then is unreliable; and against re-planning in the middle of a cycle. |
| 3 | **Workspace bounds** | Rejects detections outside the 150 mm × 150 mm rectangle: **false detections** and objects **outside the reachable / safe area** (bins, base, table edge) cannot turn into unsafe or unreachable motions. |
| 4 | **Detection buffer + wait time (2 s)** | Selects from many frames instead of the first: guards against **flicker and missed detections** (a stacked cube or a higher-priority colour seen a moment later) and makes the choice repeatable. |
| 5 | **`check_stop()` between moves** | Lets an operator **abort a sequence** (key `s` in the debug mode): the pump is switched off and the sequence ends, so a wrong movement is not completed. |
| 6 | **Idle timer (`NO_DETECTION_TIMEOUT`)** | If no detection has arrived for that time and the arm is not at home, it resets the state (`first_detection_time`, target) and returns home: guards against a **stale state** (a half-finished cycle, leftovers from lost messages) and the arm **staying** in the work area. |
| 7 | **Intermediate waypoint in `go_home`** | The way from "tool down over the bin/workspace" to the "camera pose" goes through a fixed via point: avoids **collisions** with cubes, bins and the camera mount on the return and keeps the wrist away from awkward, near-singular configurations. |

(These answer PDF question 12 as well: three concrete mechanisms, where they appear in the code, and the failure mode each guards against. For example: workspace filter in
`is_inside_workspace` called from `object_callback` and `choose_highest_priority_target`; detection buffer in `object_callback`; `check_stop` in `execute_pick_and_place`.)

In [32]:
# Space for own notes / student answers: the table above is the solution.

---
## Full integration
## Task 25: `process_detection`

In [33]:
def process_detection(mc, det_x_m, det_y_m, det_z_m, color):
    """
    Process a single detection: validate, convert, and execute pick-and-place.

    Parameters are in metres (as received from the vision node).
    Returns True if executed, False if rejected.
    """
    # Check workspace bounds (in metres)
    if not is_inside_workspace(det_x_m, det_y_m):     # validate the detected planar position
        print(f'Detection rejected: outside workspace')
        return False

    # Check colour is known
    if color not in BIN_COORDS:                        # the colour must have a configured destination
        print(f'Detection rejected: unknown colour {color}')
        return False

    # Convert metres to mm
    obj_x = det_x_m * 1000.0
    obj_y = det_y_m * 1000.0
    obj_z = det_z_m * 1000.0

    print(f'Processing: {color} at ({obj_x:.1f}, {obj_y:.1f}, {obj_z:.1f}) mm')

    # Execute
    execute_pick_and_place(mc, obj_x, obj_y, obj_z, color)
    return True

**How it works:** a chain of guards: (1) inside the workspace? (2) known colour? (3) convert units. Only then is the (slow and irreversible) motion called.
Every `return False` is a way out **before** any hardware is touched: validate first, act last. (The node uses `COLOR_PRIORITY` for the priority
filter and `BIN_COORDS` for the destination; both lists have to contain the colour.)

---
## Task 26: Test the full pipeline

In [34]:
enable_motion = SIMULATED   # real robot: a person sets True

if enable_motion:
    success = process_detection(mc, 0.150, 0.000, 0.040, 'red')
    print(f'Execution result: {success}')
else:
    # Dry run — just test validation
    print('--- Dry-run validation tests ---')
    print('Valid red:',   process_detection(mc, 0.150, 0.000, 0.040, 'red') if enable_motion else 'skipped (motion disabled)')
    print()
    print('Testing workspace validation only:')
    print('  Inside workspace:', is_inside_workspace(0.150, 0.000))
    print('  Outside (x=0.05):', is_inside_workspace(0.050, 0.000))
    print('  Colour check (red):', 'red' in BIN_COORDS)
    print('  Colour check (purple):', 'purple' in BIN_COORDS)

Processing: red at (150.0, 0.0, 40.0) mm
Pick Z: 108.0, Hover Z: 178.0
[sim   0.00 s] send_coords [150.0, 0.0, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   1.50 s] GPIO pin 20 <- 0
[sim   1.50 s] GPIO pin 21 <- 0
Pump ON
[sim   1.50 s] send_coords [150.0, 0.0, 108.0, 180.0, 0.0, 0.0] speed 50 mode None


[sim   3.00 s] send_coords [150.0, 0.0, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   4.50 s] send_coords [132.2, -147.9, 178.0, 180.0, 0.0, 0.0] speed 50 mode None
[sim   6.00 s] send_coords [132.2, -147.9, 128.0, 180.0, 0.0, 0.0] speed 50 mode None


[sim   7.50 s] GPIO pin 20 <- 1
[sim   7.80 s] GPIO pin 21 <- 1


Pump OFF
[sim   9.80 s] send_coords [132.2, -147.9, 178.0, 180.0, 0.0, 0.0] speed 50 mode None


[sim  11.30 s] send_coords [156.4, -17.2, 247.8, 179.3, 0.2, -1.4] speed 40 mode 0


[sim  15.30 s] send_coords [51.3, -63.3, 412.7, -91.8, -0.6, -90.0] speed 40 mode 0


Robot at home position
Pick-and-place complete for red object
Execution result: True


**How it works:** the template's dry-run branch is kept: in the dry run only the validation functions are called (no robot). The example below shows the
rejection paths: both return `False` and **no `send_coords`** is printed in the log.


**Example: the rejection paths, and the colour check**

```python
print(process_detection(mc, 0.050, 0.000, 0.040, 'red'))       # outside the workspace
print(process_detection(mc, 0.150, 0.000, 0.040, 'purple'))    # unknown colour
print('  Colour check (purple):', 'purple' in BIN_COORDS)
```

Output:
```text
Detection rejected: outside workspace
False
Detection rejected: unknown colour purple
False
  Colour check (purple): False
```


---
## ROS 2 integration concept
## Task 27: Three cubes, the yellow one stacked
The data flow: camera → YOLO → `solvePnP` → depth correction → camera-to-base transform → `/detected_objects` (vision node) → workspace filter, `at_home`/`busy` checks,
2 s buffer, `choose_highest_priority_target`, eight-step sequence, home (motion node).

**Scenario:** red, green and yellow cubes in the workspace; the yellow one lies on top of another cube.
1. The arm is home, `at_home = True`, `busy = False`. The vision node publishes a detection for every visible cube per frame. The lower cube of the stack is
   hidden by the yellow one (the camera sees only the top face), so the vision node reports **yellow at level 1** (z ≈ 0.07 m), red and green at level 0 (z ≈ 0.03 m).
2. The first valid detection starts the 2 s timer; all detections in that window are buffered. After it, the next message triggers the selection: yellow has the highest z, so
   it wins although red has priority 0. (`sort key = (-z, priority)`.)
3. The sequence runs: hover, pump on, descend, lift, over the yellow bin `(245.8, −150.1)`, lower, release, retreat, home via the waypoint.
4. Back home (`at_home = True`) the process repeats: the cube that was under the yellow one is now visible at level 0; the remaining detections are red, green and that cube, all at the same height, so
   the **colour priority** decides: red (0) first, then yellow (1) if the lower cube is yellow, then green (2), then cyan (3).
5. When no valid detection is left, the idle timer keeps the arm at home.

```python
# The same scenario with the selection function: repeat "choose, remove, choose ..." until nothing is left
scene = [
    {'x': 0.150, 'y': 0.030, 'z': 0.030, 'color': 'red'},
    {'x': 0.120, 'y': -0.040, 'z': 0.030, 'color': 'green'},
    {'x': 0.190, 'y': 0.000, 'z': 0.070, 'color': 'yellow'},     # stacked on ...
    {'x': 0.190, 'y': 0.000, 'z': 0.030, 'color': 'cyan'},       # ... this cube (hidden until the yellow one is gone)
]
remaining = list(scene)
hidden = scene[3]                                                  # not visible while the yellow cube covers it
order = []
while True:
    visible = [d for d in remaining if not (d is hidden and any(o['color'] == 'yellow' for o in remaining))]
    pick = choose_target(visible)
    if pick is None:
        break
    order.append((pick['color'], round(pick['z'], 3)))
    remaining.remove(pick)
print('pick order:', order)
```

Output:
```text
pick order: [('yellow', 0.07), ('red', 0.03), ('green', 0.03), ('cyan', 0.03)]
```


- `any(cond for o in remaining)` is `True` if at least one element satisfies the condition; `is` tests object identity (not equality). The loop removes each picked cube.
  The result: yellow (stack) first, then red (priority 0), green (2), and finally the cyan cube that was under the stack (3).

---
## Task 28: Summary reflection
1. **What does each stage contribute?** *Detection* (YOLO): finds which cubes are where in the image, with colour and keypoints. *Pose estimation and transform*
   (`solvePnP`, depth scale, `T_base_cam`): turns image points into a 3D position in the robot base frame, and the stack level. *Motion*: uses that position
   (in mm, with the tool offset) to execute hover, pick, carry, release, home, and sorts by colour. Each stage converts one representation into the next: pixels → camera
   coordinates → base coordinates → joint motions.
2. **Why the two-step home sequence?** After the pick and place, the tool points down; at home the tool orientation is very different (horizontal) and the position is high above.
   A direct move would interpolate all joints at once and swing the wrist and arm through unforeseen positions, possibly through cubes, bins or the camera mount. The waypoint
   forces the move to pass through a known safe pose above the workspace with the tool already turned, so the return is predictable and collision-free.
3. **Why validate detections against the workspace before commanding motion?** A detection is only an estimate; false or wrong ones (mis-classification, reflections, wrong depth scale)
   can lie anywhere. Without the check the robot would be sent to unreachable or dangerous positions: beyond the reach, into the base or the bins, or outside the area where the
   camera calibration is valid. The check is cheap, and it puts a hard limit on what a software error can make the robot do.

---
## Short questions from the PDF (preparation for the interview)
1. **The three packages and the topic.** `vision`: camera, YOLO detection, `solvePnP`, depth correction, transform to the base frame, publishes detections.
   `mycobot_msgs`: the message definitions (`DetectedObject`, `JointVelocity`). `mycobot_motion_v1`: the motion node: subscribes, filters, selects, executes the pick and place with pump
   control. They communicate over **`/detected_objects`**.
2. **`T_base_cam`:** the matrix of Task 17: $\begin{bmatrix}R_{base,cam} & t_{base,cam}\\0&1\end{bmatrix}$ with $R = [[0,1,0],[1,0,0],[0,0,-1]]$, $t = [0.170, 0, 0.39]^T$ m.
3. **Fields of `DetectedObject.msg`:** `float32 x`, `float32 y`, `float32 z` (position in the base frame, metres) and `string color`.
4. **Why a calibration step?** `solvePnP`'s depth is systematically off (camera model, keypoint noise, cube size), and the stack level is derived from depth differences of 4 cm. The
   z-scale calibration measures one cube at a known depth and derives a correction factor, so that depths are reliable to a few millimetres before the node starts publishing.
5. **Purpose of `solvePnP`:** it computes the object's pose (rotation and translation) in the **camera frame** from known 3D points on the object (the 5 keypoints of the cube's top face),
   their detected 2D image positions and the camera intrinsics: the 2D to 3D step that a single camera cannot do by itself.
6. **Depth correction and stack level:** $z_{corr} = z_{raw}\cdot f_{scale}$ with $f_{scale} = Z_{one}/z_{ref}$; then $\Delta z = z_{corr} - Z_{one}$, level $\ell = \max(0, \operatorname{round}(-\Delta z/h_{cube}))$,
   and $Z_{abs} = Z_{one} - \ell h_{cube}$ goes into the transform (Tasks 19–20).
7. **`pump_on` / `pump_off`, two pins:** `pump_on`: both pins LOW: motor on, valve closed (suction). `pump_off`: motor off, wait 0.3 s, open the valve, wait 2 s: vacuum breaks and the
   cube drops. Two pins because the motor (create vacuum) and valve (vent) are independent actuators: this gives hold, fast release and a safe off state (Task 7).
8. **Workspace limits:** to reject false detections and unreachable or unsafe targets (Task 4/24).
9. **Why buffer for 2 s?** To choose among several frames rather than the first, robust against flicker and missed detections, and to find stacked cubes / higher priority colours (Task 21).
10. **Home, hover, pick:** *home* = the fixed pose with the arm out of the way and the camera viewing the workspace (`HOME_COORDS`, z ≈ 413 mm). *Hover* = a pose 70 mm above the pick height over the object, for a safe vertical approach
    and exit (z = z_pick + 70). *Pick* = the pose in which the nozzle tip touches the cube (z = z_obj + 68), where the suction happens.
11. **Which bin?** By the detected **colour**: `BIN_COORDS[color]` (dictionary lookup) gives the `[x, y]` of the bin (Task 5).
12. **Three robustness mechanisms:** see the table in Task 24 (e.g. workspace filter, detection buffer, stop check, `busy`/`at_home` flags, idle timer, waypoint).

---
## Example: what `solvePnP` does (OpenCV)
To make Task 19's "depth from a small object" concrete: project the five known keypoints of a cube face with a known pose, then recover the pose from the projection alone.

```python
import cv2

K_cam = np.array([[756.78, 0, 327.93], [0, 751.59, 224.30], [0, 0, 1]], dtype=np.float64)
dist = np.zeros(5)                                            # ignore lens distortion in this demo
s = 0.02                                                      # half side of the cube face (m)
object_points = np.array([[-s, s, 0], [s, s, 0], [s, -s, 0], [-s, -s, 0], [0, 0, 0]], dtype=np.float64)

rvec_true = np.array([0.1, -0.05, 0.3]); tvec_true = np.array([0.02, -0.01, 0.385])
image_points, _ = cv2.projectPoints(object_points, rvec_true, tvec_true, K_cam, dist)     # what the camera would see
ok, rvec, tvec = cv2.solvePnP(object_points, image_points, K_cam, dist)
print('image points (pixels):\n', image_points.reshape(-1, 2).round(1))
print('recovered tvec:', tvec.ravel().round(4), ' true:', tvec_true)

# a cube 4 cm closer to the camera (a second level): the depth drops by 4 cm
image_points2, _ = cv2.projectPoints(object_points, rvec_true, tvec_true - [0, 0, 0.04], K_cam, dist)
ok, rvec2, tvec2 = cv2.solvePnP(object_points, image_points2, K_cam, dist)
print('depth tz: %.4f m -> %.4f m (level up = 4 cm closer)' % (tvec[2, 0], tvec2[2, 0]))
```

Output:
```text
image points (pixels):
 [[318.1 230.5]
 [392.5 253.1]
 [416.6 179. ]
 [341.5 155.7]
 [367.2 204.8]]
recovered tvec: [ 0.02  -0.01   0.385]  true: [ 0.02  -0.01   0.385]
depth tz: 0.3850 m -> 0.3450 m (level up = 4 cm closer)
```


- `cv2.projectPoints` = the camera model: 3D points + pose + intrinsics → pixel coordinates. `cv2.solvePnP` inverts it: from 5 correspondences (3D model point ↔ pixel) it recovers rotation (`rvec`, Rodrigues vector) and translation
  (`tvec`). Here it reproduces the true depth (0.385 m) exactly, because the demo has no noise and no distortion.
- In the real system the keypoints come from YOLO (noisy, ± a few pixels), the intrinsics come from a camera calibration and the cube size is assumed: all errors that the z-scale factor absorbs.
- The depth changes by exactly the cube height for a stack level, which is the principle of the level detection.

---
## Cleanup (real hardware only)
On the Pi at the end of the session: `GPIO.cleanup()` releases the pins (pins back to inputs, pump driver off), and the port must be released (`del mc` or closing the kernel) so that other notebooks
can use it again.

In [38]:
# Space for student answers: see the reflection above.

# Optional cleanup (real hardware, end of the session):
# GPIO.cleanup()

---
## Summary
- Geometry: `T_base_cam` (camera 17 cm ahead, 39 cm up, looking down), position in the base frame = R·p_cam + t; depth scale $f = Z_{one}/z_{ref}$; stack level = round(−Δz/4 cm).
- Motion: everything in mm/degrees, tool pointing down `[180, 0, 0]`; heights = object + 68 (nozzle) [+ 70 hover]; drop at 60 + 68; via the waypoint to home.
- Selection: filter (colour known, inside workspace) → sort by (−z, priority) → first.
- Safety: guard flags (`enable_*`), stop checks between moves, validation before action, safe default pin state, waypoint, workspace bounds.
- This notebook ran in simulation: **no** real pump or motion command was issued.